<!-- @format -->

# Baseline: YOLOv8n Pretrained (COCO) - Không Train

**Mục tiêu:** Tạo baseline bằng model YOLOv8n pretrained trên COCO (80 classes) để so sánh với model đã train trên dataset helmet.

**Lưu ý quan trọng:**

- YOLOv8n pretrained trên COCO **không có class `helmet` hay `no helmet`**
- COCO chỉ có 80 class chung (person, car, bicycle, ...)
- Baseline này chỉ detect được **`person`** — không phân biệt được đội mũ hay không
- Mục đích: cho thấy sự khác biệt giữa model chưa train vs đã train trên task cụ thể

| Model                   | Classes                       | Phân biệt helmet? |
| ----------------------- | ----------------------------- | ----------------- |
| YOLOv8n COCO (baseline) | 80 classes (person, car, ...) | ❌ Không          |
| YOLOv8n Trained         | 2 classes (helmet, no helmet) | ✅ Có             |


<!-- @format -->

## 1. Import & Cấu hình


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import torch

BASE_DIR = Path(r"D:\Project_CV\Helmet_Detection\ai-model")
DATA_YAML = BASE_DIR / "data4training" / "data.yaml"
TEST_IMG_DIR = BASE_DIR / "data4training" / "images" / "test"
TRAINED_MODEL_PATH = BASE_DIR / "runs" / "helmet_detect" / "weights" / "best.pt"

IMG_SIZE = 640
CONF = 0.25

print(f"Test images: {len(list(TEST_IMG_DIR.glob('*.*')))} ảnh")
print(f"Trained model exists: {TRAINED_MODEL_PATH.exists()}")

<!-- @format -->

## 2. Load cả 2 model: Baseline (COCO) vs Trained


In [ ]:
# Baseline: YOLOv8n pretrained trên COCO (không train gì thêm)
baseline_model = YOLO("yolov8n.pt")
print(f"Baseline model: YOLOv8n COCO — {len(baseline_model.names)} classes")
print(f"  Một số classes: {dict(list(baseline_model.names.items())[:10])}")

# Trained model: đã train trên helmet dataset
trained_model = YOLO(str(TRAINED_MODEL_PATH))
print(f"\nTrained model: {len(trained_model.names)} classes")
print(f"  Classes: {trained_model.names}")

<!-- @format -->

## 3. So sánh kết quả Predict trên cùng tập ảnh test


In [ ]:
# Lấy 6 ảnh test
test_images = sorted(TEST_IMG_DIR.glob("*.*"))[:6]

fig, axes = plt.subplots(len(test_images), 2, figsize=(16, 5 * len(test_images)))

for idx, img_path in enumerate(test_images):
    img_str = str(img_path)
    
    # --- Baseline (COCO) — chỉ giữ class 'person' (class 0) ---
    baseline_results = baseline_model.predict(
        source=img_str, imgsz=IMG_SIZE, conf=CONF, classes=[0], verbose=False
    )
    baseline_img = baseline_results[0].plot(line_width=2, font_size=12)
    baseline_img_rgb = baseline_img[:, :, ::-1]
    baseline_count = len(baseline_results[0].boxes)
    
    # --- Trained model ---
    trained_results = trained_model.predict(
        source=img_str, imgsz=IMG_SIZE, conf=CONF, verbose=False
    )
    trained_img = trained_results[0].plot(line_width=2, font_size=12)
    trained_img_rgb = trained_img[:, :, ::-1]
    
    helmet_count = sum(1 for b in trained_results[0].boxes if int(b.cls[0]) == 0)
    no_helmet_count = sum(1 for b in trained_results[0].boxes if int(b.cls[0]) == 1)
    
    # Vẽ
    axes[idx, 0].imshow(baseline_img_rgb)
    axes[idx, 0].set_title(f"BASELINE (COCO)\nPerson: {baseline_count}", fontsize=11, color='gray')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(trained_img_rgb)
    axes[idx, 1].set_title(
        f"TRAINED\n✅ Helmet: {helmet_count} | ❌ No helmet: {no_helmet_count}",
        fontsize=11, color='green' if no_helmet_count == 0 else 'red'
    )
    axes[idx, 1].axis('off')

plt.suptitle("So sánh: Baseline (COCO - chỉ detect person) vs Trained (helmet/no helmet)",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

<!-- @format -->

## 4. Đánh giá Baseline trên tập Test

Vì baseline COCO không có class `helmet`/`no helmet`, ta **không thể chạy `model.val()` trên data.yaml** (khác format label).

Thay vào đó, ta đánh giá baseline = **"mọi người detect được đều coi là helmet"** (best-case cho baseline) để xem mAP sẽ thấp thế nào.


In [ ]:
# Đánh giá trained model bằng val() chính thức
print("=" * 60)
print("ĐÁNH GIÁ TRAINED MODEL TRÊN TẬP TEST")
print("=" * 60)

metrics = trained_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=16,
    verbose=False,
)

print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")

class_names = ["helmet", "no helmet"]
print(f"\n{'Class':<15} {'Precision':<12} {'Recall':<12} {'mAP50':<12}")
print("-" * 51)
for i, name in enumerate(class_names):
    print(f"{name:<15} {metrics.box.p[i]:.4f}       {metrics.box.r[i]:.4f}       {metrics.box.ap50[i]:.4f}")

In [ ]:
# Thống kê baseline: detect person trên tất cả ảnh test
print("=" * 60)
print("THỐNG KÊ BASELINE (COCO) TRÊN TẬP TEST")
print("=" * 60)

all_test_images = sorted(TEST_IMG_DIR.glob("*.*"))
total_persons = 0
images_with_person = 0

for img_path in all_test_images:
    results = baseline_model.predict(
        source=str(img_path), imgsz=IMG_SIZE, conf=CONF, classes=[0], verbose=False
    )
    n = len(results[0].boxes)
    total_persons += n
    if n > 0:
        images_with_person += 1

print(f"  Tổng ảnh test:          {len(all_test_images)}")
print(f"  Ảnh có detect person:   {images_with_person}")
print(f"  Tổng persons detected:  {total_persons}")
print(f"\n⚠️  Baseline COCO chỉ biết 'có người' — KHÔNG phân biệt được helmet vs no helmet")
print(f"    → mAP cho task helmet detection = 0 (không applicable)")

<!-- @format -->

## 5. Bảng so sánh tổng kết


In [ ]:
print("\n" + "=" * 70)
print("BẢNG SO SÁNH: BASELINE vs TRAINED MODEL")
print("=" * 70)
print(f"{'Tiêu chí':<30} {'Baseline (COCO)':<20} {'Trained':<20}")
print("-" * 70)
print(f"{'Model':<30} {'YOLOv8n COCO':<20} {'YOLOv8n fine-tuned':<20}")
print(f"{'Training':<30} {'Không (pretrained)':<20} {'50 epochs':<20}")
print(f"{'Số classes':<30} {'80 (COCO)':<20} {'2 (helmet/no)':<20}")
print(f"{'Detect helmet?':<30} {'❌ Không':<20} {'✅ Có':<20}")
print(f"{'Detect no helmet?':<30} {'❌ Không':<20} {'✅ Có':<20}")
print(f"{'mAP50 (helmet task)':<30} {'N/A (0.0000)':<20} {metrics.box.map50:<20.4f}")
print(f"{'mAP50-95 (helmet task)':<30} {'N/A (0.0000)':<20} {metrics.box.map:<20.4f}")
print(f"{'Precision':<30} {'N/A':<20} {metrics.box.mp:<20.4f}")
print(f"{'Recall':<30} {'N/A':<20} {metrics.box.mr:<20.4f}")
print("-" * 70)
print("\n📌 KẾT LUẬN:")
print("   YOLOv8n pretrained COCO KHÔNG THỂ nhận diện helmet/no helmet vì")
print("   dataset COCO không chứa các class này.")
print("   → Cần fine-tune trên dataset helmet riêng để model học được task cụ thể.")
print(f"   → Sau khi train: mAP50 đạt {metrics.box.map50:.4f} (từ 0.0 lên {metrics.box.map50:.4f})")